# Feature Engineering

* 8 joint angles
* 12 axis angles

### Imports

In [15]:
import numpy as np
import pandas as pd
from pathlib import Path

### Configuration

In [16]:
IN_MOVENET_DIR = Path("../data/processed/keypoints_augmented/movenet")
IN_MEDIAPIPE_NORM_DIR = Path("../data/processed/keypoints_augmented/mediapipe_norm")
IN_MEDIAPIPE_WORLD_DIR = Path("../data/processed/keypoints_augmented/mediapipe_world")

OUT_MOVENET_DIR = Path("../data/processed/keypoints_augmented_angles/movenet")
OUT_MEDIAPIPE_NORM_DIR = Path("../data/processed/keypoints_augmented_angles/mediapipe_norm")
OUT_MEDIAPIPE_WORLD_DIR = Path("../data/processed/keypoints_augmented_angles/mediapipe_world")

IN_MOVENET_ORIG_DIR = Path("../data/processed/keypoints/movenet")
IN_MEDIAPIPE_NORM_ORIG_DIR = Path("../data/processed/keypoints/mediapipe_norm")
IN_MEDIAPIPE_WORLD_ORIG_DIR = Path("../data/processed/keypoints/mediapipe_world")

OUT_MOVENET_ORIG_DIR = Path("../data/processed/keypoints_angles/movenet")
OUT_MEDIAPIPE_NORM_ORIG_DIR = Path("../data/processed/keypoints_angles/mediapipe_norm")
OUT_MEDIAPIPE_WORLD_ORIG_DIR = Path("../data/processed/keypoints_angles/mediapipe_world")

EPS = 1e-8

JOINTS = [
    "left_shoulder", "right_shoulder",
    "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
    "left_hip", "right_hip",
    "left_knee", "right_knee",
    "left_ankle", "right_ankle",
]

### Angle calculation

In [17]:
def angle_abc(points_a, points_b, points_c):
    u = points_a - points_b
    v = points_c - points_b
    du = np.linalg.norm(u, axis=1)
    dv = np.linalg.norm(v, axis=1)
    denom = du * dv
    dot = np.einsum("ij,ij->i", u, v)

    out = np.zeros(points_a.shape[0], dtype=points_a.dtype)
    ok = denom > EPS
    cos_theta = np.zeros_like(dot)
    cos_theta[ok] = dot[ok] / denom[ok]
    cos_theta[ok] = np.clip(cos_theta[ok], -1.0, 1.0)
    out[ok] = np.arccos(cos_theta[ok])
    return out

def angle_segment_to_axis(p0, p1, axis_unit):
    v = p1 - p0
    dv = np.linalg.norm(v, axis=1)

    out = np.zeros(p0.shape[0], dtype=p0.dtype)
    ok = dv > EPS
    dot = v @ axis_unit
    cos_theta = np.zeros(p0.shape[0], dtype=p0.dtype)
    cos_theta[ok] = dot[ok] / dv[ok]
    cos_theta[ok] = np.clip(cos_theta[ok], -1.0, 1.0)
    out[ok] = np.arccos(cos_theta[ok])
    return out

### Helpers

In [18]:
def load_npz_as_arrays(npz_path):
    f = np.load(npz_path, allow_pickle=False)
    data = f["data"]
    cols = f["columns"].astype(str)
    return data, cols

def load_csv_as_arrays(csv_path):
    df = pd.read_csv(csv_path)
    cols = df.columns.astype(str).to_numpy()
    data = df.to_numpy()
    return data, cols

def load_file_as_arrays(path):
    suf = path.suffix.lower()
    if suf == ".npz":
        return load_npz_as_arrays(path)
    if suf == ".csv":
        return load_csv_as_arrays(path)
    raise ValueError(f"Unsupported file type: {path}")

def build_col_index(cols):
    return {c: i for i, c in enumerate(cols)}

def get_xy(data, idx, joint):
    return np.stack([data[:, idx[f"{joint}_x"]], data[:, idx[f"{joint}_y"]]], axis=1)

def get_xyz(data, idx, joint):
    return np.stack(
        [data[:, idx[f"{joint}_x"]], data[:, idx[f"{joint}_y"]], data[:, idx[f"{joint}_z"]]],
        axis=1
    )

### Feature computation

In [19]:
def compute_angle_features_2d(data, cols):
    idx = build_col_index(cols)
    P = {j: get_xy(data, idx, j) for j in JOINTS}

    y_axis = np.array([0.0, 1.0], dtype=data.dtype)
    x_axis = np.array([1.0, 0.0], dtype=data.dtype)

    feats = []
    feat_names = []

    # 8 joint angles
    feats += [angle_abc(P["left_shoulder"], P["left_elbow"], P["left_wrist"])]
    feat_names += ["ang_left_elbow"]
    feats += [angle_abc(P["right_shoulder"], P["right_elbow"], P["right_wrist"])]
    feat_names += ["ang_right_elbow"]

    feats += [angle_abc(P["left_hip"], P["left_shoulder"], P["left_elbow"])]
    feat_names += ["ang_left_shoulder"]
    feats += [angle_abc(P["right_hip"], P["right_shoulder"], P["right_elbow"])]
    feat_names += ["ang_right_shoulder"]

    feats += [angle_abc(P["left_shoulder"], P["left_hip"], P["left_knee"])]
    feat_names += ["ang_left_hip"]
    feats += [angle_abc(P["right_shoulder"], P["right_hip"], P["right_knee"])]
    feat_names += ["ang_right_hip"]

    feats += [angle_abc(P["left_hip"], P["left_knee"], P["left_ankle"])]
    feat_names += ["ang_left_knee"]
    feats += [angle_abc(P["right_hip"], P["right_knee"], P["right_ankle"])]
    feat_names += ["ang_right_knee"]

    # 12 axis angles
    feats += [angle_segment_to_axis(P["left_hip"], P["left_shoulder"], y_axis)]
    feat_names += ["ang_left_hip_shoulder_y"]
    feats += [angle_segment_to_axis(P["right_hip"], P["right_shoulder"], y_axis)]
    feat_names += ["ang_right_hip_shoulder_y"]

    feats += [angle_segment_to_axis(P["left_shoulder"], P["left_elbow"], y_axis)]
    feat_names += ["ang_left_shoulder_elbow_y"]
    feats += [angle_segment_to_axis(P["right_shoulder"], P["right_elbow"], y_axis)]
    feat_names += ["ang_right_shoulder_elbow_y"]

    feats += [angle_segment_to_axis(P["left_elbow"], P["left_wrist"], y_axis)]
    feat_names += ["ang_left_elbow_wrist_y"]
    feats += [angle_segment_to_axis(P["right_elbow"], P["right_wrist"], y_axis)]
    feat_names += ["ang_right_elbow_wrist_y"]

    feats += [angle_segment_to_axis(P["left_hip"], P["left_knee"], y_axis)]
    feat_names += ["ang_left_hip_knee_y"]
    feats += [angle_segment_to_axis(P["right_hip"], P["right_knee"], y_axis)]
    feat_names += ["ang_right_hip_knee_y"]

    feats += [angle_segment_to_axis(P["left_knee"], P["left_ankle"], y_axis)]
    feat_names += ["ang_left_knee_ankle_y"]
    feats += [angle_segment_to_axis(P["right_knee"], P["right_ankle"], y_axis)]
    feat_names += ["ang_right_knee_ankle_y"]

    feats += [angle_segment_to_axis(P["left_hip"], P["right_hip"], x_axis)]
    feat_names += ["ang_hips_x"]
    feats += [angle_segment_to_axis(P["left_shoulder"], P["right_shoulder"], x_axis)]
    feat_names += ["ang_shoulders_x"]

    feat_mat = np.stack(feats, axis=1)
    return feat_mat, np.array(feat_names, dtype=object)

def compute_angle_features_3d_world(data, cols):
    idx = build_col_index(cols)
    P = {j: get_xyz(data, idx, j) for j in JOINTS}

    y_axis = np.array([0.0, 1.0, 0.0], dtype=data.dtype)
    x_axis = np.array([1.0, 0.0, 0.0], dtype=data.dtype)

    feats = []
    feat_names = []

    # 8 joint angles
    feats += [angle_abc(P["left_shoulder"], P["left_elbow"], P["left_wrist"])]
    feat_names += ["ang_left_elbow"]
    feats += [angle_abc(P["right_shoulder"], P["right_elbow"], P["right_wrist"])]
    feat_names += ["ang_right_elbow"]

    feats += [angle_abc(P["left_hip"], P["left_shoulder"], P["left_elbow"])]
    feat_names += ["ang_left_shoulder"]
    feats += [angle_abc(P["right_hip"], P["right_shoulder"], P["right_elbow"])]
    feat_names += ["ang_right_shoulder"]

    feats += [angle_abc(P["left_shoulder"], P["left_hip"], P["left_knee"])]
    feat_names += ["ang_left_hip"]
    feats += [angle_abc(P["right_shoulder"], P["right_hip"], P["right_knee"])]
    feat_names += ["ang_right_hip"]

    feats += [angle_abc(P["left_hip"], P["left_knee"], P["left_ankle"])]
    feat_names += ["ang_left_knee"]
    feats += [angle_abc(P["right_hip"], P["right_knee"], P["right_ankle"])]
    feat_names += ["ang_right_knee"]

    # 12 axis angles
    feats += [angle_segment_to_axis(P["left_hip"], P["left_shoulder"], y_axis)]
    feat_names += ["ang_left_hip_shoulder_y"]
    feats += [angle_segment_to_axis(P["right_hip"], P["right_shoulder"], y_axis)]
    feat_names += ["ang_right_hip_shoulder_y"]

    feats += [angle_segment_to_axis(P["left_shoulder"], P["left_elbow"], y_axis)]
    feat_names += ["ang_left_shoulder_elbow_y"]
    feats += [angle_segment_to_axis(P["right_shoulder"], P["right_elbow"], y_axis)]
    feat_names += ["ang_right_shoulder_elbow_y"]

    feats += [angle_segment_to_axis(P["left_elbow"], P["left_wrist"], y_axis)]
    feat_names += ["ang_left_elbow_wrist_y"]
    feats += [angle_segment_to_axis(P["right_elbow"], P["right_wrist"], y_axis)]
    feat_names += ["ang_right_elbow_wrist_y"]

    feats += [angle_segment_to_axis(P["left_hip"], P["left_knee"], y_axis)]
    feat_names += ["ang_left_hip_knee_y"]
    feats += [angle_segment_to_axis(P["right_hip"], P["right_knee"], y_axis)]
    feat_names += ["ang_right_hip_knee_y"]

    feats += [angle_segment_to_axis(P["left_knee"], P["left_ankle"], y_axis)]
    feat_names += ["ang_left_knee_ankle_y"]
    feats += [angle_segment_to_axis(P["right_knee"], P["right_ankle"], y_axis)]
    feat_names += ["ang_right_knee_ankle_y"]

    feats += [angle_segment_to_axis(P["left_hip"], P["right_hip"], x_axis)]
    feat_names += ["ang_hips_x"]
    feats += [angle_segment_to_axis(P["left_shoulder"], P["right_shoulder"], x_axis)]
    feat_names += ["ang_shoulders_x"]

    feat_mat = np.stack(feats, axis=1)
    return feat_mat, np.array(feat_names, dtype=object)

### Processing

In [ ]:
def process_directory(in_dir, out_dir, mode):
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(list(in_dir.glob("*.npz")) + list(in_dir.glob("*.csv")))
    for i, path in enumerate(files, 1):
        data, cols = load_file_as_arrays(path)

        if mode in ("2d", "norm3d_as_2d"):
            feat_mat, feat_names = compute_angle_features_2d(data, cols)
        elif mode == "3d_world":
            feat_mat, feat_names = compute_angle_features_3d_world(data, cols)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        new_data = np.concatenate([data, feat_mat], axis=1)
        new_cols = np.concatenate([cols.astype(object), feat_names], axis=0)

        out_path = out_dir / f"{path.stem}_angles.npz"
        np.savez_compressed(out_path, data=new_data, columns=new_cols)

        if i % 500 == 0 or i == len(files):
            print(f"[{in_dir.name}] {i}/{len(files)} done")

process_directory(IN_MOVENET_DIR, OUT_MOVENET_DIR, mode="2d")
process_directory(IN_MEDIAPIPE_NORM_DIR, OUT_MEDIAPIPE_NORM_DIR, mode="norm3d_as_2d")
process_directory(IN_MEDIAPIPE_WORLD_DIR, OUT_MEDIAPIPE_WORLD_DIR, mode="3d_world")

process_directory(IN_MOVENET_ORIG_DIR, OUT_MOVENET_ORIG_DIR, mode="2d")
process_directory(IN_MEDIAPIPE_NORM_ORIG_DIR, OUT_MEDIAPIPE_NORM_ORIG_DIR, mode="norm3d_as_2d")
process_directory(IN_MEDIAPIPE_WORLD_ORIG_DIR, OUT_MEDIAPIPE_WORLD_ORIG_DIR, mode="3d_world")

[movenet] 500/10230 done
[movenet] 1000/10230 done
[movenet] 1500/10230 done
[movenet] 2000/10230 done
[movenet] 2500/10230 done
[movenet] 3000/10230 done
[movenet] 3500/10230 done
[movenet] 4000/10230 done
[movenet] 4500/10230 done
[movenet] 5000/10230 done
[movenet] 5500/10230 done
[movenet] 6000/10230 done
[movenet] 6500/10230 done
[movenet] 7000/10230 done
[movenet] 7500/10230 done
[movenet] 8000/10230 done
[movenet] 8500/10230 done
[movenet] 9000/10230 done
[movenet] 9500/10230 done
[movenet] 10000/10230 done
[movenet] 10230/10230 done
[mediapipe_norm] 500/10230 done
[mediapipe_norm] 1000/10230 done
[mediapipe_norm] 1500/10230 done
[mediapipe_norm] 2000/10230 done
[mediapipe_norm] 2500/10230 done
[mediapipe_norm] 3000/10230 done
[mediapipe_norm] 3500/10230 done
[mediapipe_norm] 4000/10230 done
[mediapipe_norm] 4500/10230 done
[mediapipe_norm] 5000/10230 done
[mediapipe_norm] 5500/10230 done
[mediapipe_norm] 6000/10230 done
[mediapipe_norm] 6500/10230 done
[mediapipe_norm] 7000/102

In [22]:
def convert_npz_dir_to_csv(in_dir):
    npz_files = sorted(in_dir.glob("*.npz"))
    for path in npz_files:
        f = np.load(path, allow_pickle=True)
        df = pd.DataFrame(f["data"], columns=f["columns"])
        df.to_csv(path.with_suffix(".csv"), index=False)
        path.unlink()
    print(f"{in_dir.name}: {len(npz_files)} files converted")

convert_npz_dir_to_csv(OUT_MOVENET_ORIG_DIR)
convert_npz_dir_to_csv(OUT_MEDIAPIPE_NORM_ORIG_DIR)
convert_npz_dir_to_csv(OUT_MEDIAPIPE_WORLD_ORIG_DIR)

movenet: 10 files converted
mediapipe_norm: 10 files converted
mediapipe_world: 10 files converted
